<a href="https://colab.research.google.com/github/E-Sentinel-Project/E-Sentinel/blob/master/BatteryStats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import display
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept='.txt',
    multiple=False
)

display(uploader)


FileUpload(value={}, accept='.txt', description='Upload')

In [ ]:
import re

APP_PACKAGE = "com.example.e_sentinel"

uploaded_file = list(uploader.value.values())[0]
content = uploaded_file['content'].decode('utf-8', errors='ignore')

# Regex
time_pattern = re.compile(r"\+(\d+)m(\d+)s(\d+)ms")
charge_pattern = re.compile(r"charge=(\d+)")
app_pattern = re.compile(APP_PACKAGE)

timestamps = []
charges = []
inside_app_window = False

for line in content.splitlines():

    # Detect when E-Sentinel comes to foreground
    if f'+top=' in line and APP_PACKAGE in line:
        inside_app_window = True

    # Detect when E-Sentinel leaves foreground
    if f'-top=' in line and APP_PACKAGE in line:
        inside_app_window = False

    # Capture battery only when app is active
    if inside_app_window:
        t_match = time_pattern.search(line)
        c_match = charge_pattern.search(line)

        if t_match and c_match:
            minutes = int(t_match.group(1))
            seconds = int(t_match.group(2))
            millis  = int(t_match.group(3))

            total_seconds = minutes * 60 + seconds + millis / 1000
            charge = int(c_match.group(1))

            timestamps.append(total_seconds)
            charges.append(charge)

# Validation
if len(charges) < 2:
    print("Not enough E-Sentinel battery data found.")
else:
    initial_charge = charges[0]
    final_charge = charges[-1]

    battery_used = final_charge - initial_charge
    total_time_hr = (timestamps[-1] - timestamps[0]) / 3600
    avg_consumption = battery_used / total_time_hr if total_time_hr > 0 else 0

    print("----- E-Sentinel Battery Consumption -----")
    print(f"App Package         : {APP_PACKAGE}")
    print(f"Battery Used        : {battery_used:.2f} mAh")
    print(f"Active Time         : {total_time_hr:.2f} hours")
    print(f"Average Consumption : {avg_consumption:.2f} mAh/hour")


----- E-Sentinel Battery Consumption -----
App Package         : com.example.e_sentinel
Battery Used        : 171.00 mAh
Active Time         : 0.79 hours
Average Consumption : 215.51 mAh/hour
